# Extracción y Limpieza de PDFs

Procesamiento de archivos PDF para sistemas RAG: extracción de texto, limpieza avanzada de caracteres y metadatos.

In [15]:
import os
import re
import json
import unicodedata
from pathlib import Path
from typing import List, Dict

from PyPDF2 import PdfReader
import pandas as pd
 
# CONSTANTES
# Límites numéricos de procesamiento
MAX_TEXT_LENGTH = 1000  # Máximo de caracteres para analizar
MAX_LINES_TO_CHECK = 20  # Máximo de líneas para buscar nombres
MIN_WORD_LENGTH = 1     # Longitud mínima de palabra
MIN_NAME_WORDS = 2      # Mínimo de palabras en un nombre
MAX_NAME_WORDS = 4      # Máximo de palabras en un nombre

# Caracteres especiales latinos
SPECIAL_CHARS = {
    'ı': 'i',  # i sin punto (latín)
    'ȷ': 'j',  # j sin punto (latín)
    'ø': 'o',  # o con barra
    'Ø': 'O',  # O con barra
    'ß': 'ss', # s alemana
}

# Reemplazos de caracteres especiales
CHAR_REPLACEMENTS = {
    'ñ': 'n', 'Ñ': 'N',
    'ç': 'c', 'Ç': 'C',
    '¡': '', '¿': '',
    '«': '"', '»': '"',
    ''': "'", ''': "'", 
    '"': '"', '"': '"',
    '–': '-', '—': '-',
    '…': '...'
}

# Patrones de acentos mal formateados
ACCENT_PATTERNS = [
    # Para consonante + espacio + acento + vocal
    (r'([bcdfghjklmnpqrstvwxyzBCDFGHJKLMNPQRSTVWXYZ])\s+´([aeiouAEIOU])', r'\1\2'),
    (r'([bcdfghjklmnpqrstvwxyzBCDFGHJKLMNPQRSTVWXYZ])\s+`([aeiouAEIOU])', r'\1\2'), 
    (r'([bcdfghjklmnpqrstvwxyzBCDFGHJKLMNPQRSTVWXYZ])\s+¨([aeiouAEIOU])', r'\1\2'),
    (r'([bcdfghjklmnpqrstvwxyzBCDFGHJKLMNPQRSTVWXYZ])\s+˜([nN])', r'\1\2'),
    # Para espacios + acento + vocal (casos generales)
    (r'\s+´([aeiouAEIOU])', r'\1'),
    (r'\s+`([aeiouAEIOU])', r'\1'),
    (r'\s+¨([aeiouAEIOU])', r'\1'),
    (r'\s+˜([nN])', r'\1'),
    # Casos específicos problemáticos
    (r'\s+´i', 'i'),
    (r'\s+´a', 'a'),
    (r'\s+´e', 'e'),
    (r'\s+´o', 'o'),
    (r'\s+´u', 'u'),
]

# Patrones para detección de autores
AUTHOR_PATTERNS = [
    # Patrón 1: Después de palabras clave "Por:", "Autor:", etc. (2-4 palabras)
    r'(?:Por|Autor|Author|By):\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3})',
    # Patrón 2: Nombres cerca de códigos de estudiante o emails (2-4 palabras)
    r'([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3})\s*(?:\d{7,}|[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})',
    # Patrón 3: Nombres con apellidos seguidos de contexto institucional específico
    r'\b([A-Z][a-z]+\s+[A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,2})\s+(?:Escuela|Instituto|Universidad|Facultad|Carne|Carnet)\b',
    # Patrón 4: Nombres completos precedidos por contexto académico específico
    r'(?:Estudiante|Alumno|Student)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3})\s+(?:Escuela|Instituto|Universidad|Carne|\d)',
]

# Listas de palabras clave para filtrar
INSTITUTIONAL_KEYWORDS = [
    'universidad', 'instituto', 'tecnologico', 'escuela', 'clase', 'notas', 
    'semana', 'inteligencia', 'artificial', 'computacion', 'costa', 'rica', 
    'cartago'
]

ACADEMIC_KEYWORDS = [
    'notas', 'clase', 'semana', 'octubre', 'inteligencia', 'artificial',
    'universidad', 'instituto', 'escuela', 'tecnologico', 'computacion',
    'costa', 'rica', 'cartago', 'proyecto', 'final', 'redes', 'neuronales', 
    'modelos', 'lenguajes', 'extensos', 'ingenieria', 'convolucionales',
    'retrieval', 'augmented', 'generation', 'index', 'terms', 'machine',
    'learning', 'algebra', 'lineal', 'using', 'visual', 'studio', 'code',
    'small', 'language', 'regresion', 'validation', 'apuntes', 'verosimilitud',
    'hinge', 'loss', 'estudiante', 'aprendizaje', 'supervisado', 'pacheco',
    'portuguez', 'fold', 'cross', 'autonomous', 'agents', 'users', 'tools'
]

## Configuración de Rutas

In [16]:
# Configuración de rutas
docs_path = Path("./docs")
output_path = Path("./data/clean_texts")
output_path.mkdir(parents=True, exist_ok=True)
output_metadata_path = Path("./data/metadata")
output_metadata_path.mkdir(parents=True, exist_ok=True)

## Función de Limpieza de Texto

In [17]:
def clean_text(text: str) -> str:
    """
    Limpia texto eliminando caracteres no deseados y normalizando espacios.
    Maneja específicamente el problema de acentos mal formateados en PDFs.
    """
    # Reemplazar saltos de línea por espacios
    text = text.replace('\n', ' ')
    
    # Primero, manejar caracteres especiales de vocales sin punto
    for char, replacement in SPECIAL_CHARS.items():
        text = text.replace(char, replacement)
    
    # Corregir patrones específicos de acentos mal extraídos del PDF
    for pattern, replacement in ACCENT_PATTERNS:
        text = re.sub(pattern, replacement, text)
    
    # Normalizar caracteres Unicode (descomponer acentos y eliminar marcas diacríticas)
    text = unicodedata.normalize('NFD', text)
    text = ''.join(char for char in text if unicodedata.category(char) != 'Mn')
    
    # Reemplazar caracteres especiales restantes con sus equivalentes ASCII
    for char, replacement in CHAR_REPLACEMENTS.items():
        text = text.replace(char, replacement)
    
    # Eliminar cualquier caracter no ASCII restante
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    
    # Normalizar espacios múltiples
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

## Extracción de Metadatos

## Extracción de PDF

In [18]:
def extract_author_from_text(clean_text_content: str) -> str:
    """
    Algoritmo de extracción de autores optimizado:
    1. Buscar patrones de nombres propios
    2. Filtrar preposiciones/artículos  
    3. Remover palabras académicas/institucionales
    4. Seleccionar mejor candidato
    """
    if not clean_text_content:
        return "Desconocido"
    
    search_text = clean_text_content[:MAX_TEXT_LENGTH]
    
    # PASO 1: Encontrar patrones de nombres propios
    nombre_propio_pattern = r'\b([A-Z][a-z]{2,}(?:\s+[A-Z][a-z]{2,})*)\b'
    candidatos_raw = []
    
    matches = re.finditer(nombre_propio_pattern, search_text)
    for match in matches:
        candidato = match.group(1).strip()
        if len(candidato.split()) >= 2:  # Al menos 2 palabras
            candidatos_raw.append(candidato)
    
    if not candidatos_raw:
        return "Desconocido"
    
    # PASO 2: Excluir preposiciones y artículos
    preposiciones = {'de', 'del', 'la', 'el', 'y', 'en', 'con', 'por', 'para', 'desde', 'hasta'}
    candidatos_sin_preposiciones = []
    
    for candidato in candidatos_raw:
        palabras = candidato.split()
        palabras_limpias = [palabra for palabra in palabras 
                           if palabra.lower() not in preposiciones]
        
        if len(palabras_limpias) >= 2:
            candidato_limpio = ' '.join(palabras_limpias)
            candidatos_sin_preposiciones.append(candidato_limpio)
    
    if not candidatos_sin_preposiciones:
        return "Desconocido"
    
    # PASO 3: Filtrar palabras académicas/institucionales
    all_keywords = set(ACADEMIC_KEYWORDS + INSTITUTIONAL_KEYWORDS)
    candidatos_finales = []
    
    for candidato in candidatos_sin_preposiciones:
        palabras = candidato.split()
        palabras_finales = [palabra for palabra in palabras 
                           if palabra.lower() not in all_keywords]
        
        if len(palabras_finales) >= 2:  # Preferir nombres completos
            candidato_final = ' '.join(palabras_finales)
            candidatos_finales.append((candidato_final, len(palabras_finales)))
    
    if not candidatos_finales:
        return "Desconocido"
    
    # PASO 4: Seleccionar mejor candidato (más palabras = más completo)
    candidatos_finales.sort(key=lambda x: x[1], reverse=True)
    return candidatos_finales[0][0]

In [19]:
def extract_text_from_pdf(pdf_path: Path) -> Dict:
    """
    Extrae texto y metadatos básicos de un archivo PDF.
    FASE 1: Solo extrae texto y título, autor se establecerá en "Desconocido" para extracción posterior.
    """
    reader = PdfReader(pdf_path)
    text_pages = []
    for i, page in enumerate(reader.pages):
        try:
            text = page.extract_text() or ""
        except Exception as e:
            print(f"Error en página {i} de {pdf_path.name}: {e}")
            text = ""
        text_pages.append({
            "page": i + 1,
            "text": text
        })

    metadata = reader.metadata or {}
    
    # Extraer solo el título en este paso
    pdf_title = metadata.get('/Title', '').strip()
    
    # Si no hay título en metadatos, usar el nombre del archivo
    if not pdf_title or pdf_title == pdf_path.stem:
        title = pdf_path.stem
    else:
        title = pdf_title
    
    # Establecer autor como "Desconocido" para extracción posterior desde JSON
    author = "Desconocido"
    
    return {
        "file_name": pdf_path.name,
        "num_pages": len(text_pages),
        "metadata": {
            "author": author,
            "title": title
        },
        "pages": text_pages
    }

In [20]:
def extract_file_metadata(filename: str) -> Dict:
    """
    Extrae información del nombre del archivo basándose en el patrón de guiones bajos.
    Formato: <numero_semana>_<cadena>_<cadena>_<fecha_clase>_<numero_apunte>+<lo_que_sea>
    Ejemplos: 
    - 6_Semana_AI_20250909_2-220676337.pdf
    - 10_SEMANA_AI_20251007_1-222887296.pdf
    - 8_Semana_AI_20250923_2.pdf
    """
    import re
    from datetime import datetime
    
    # Dividir por guiones bajos para extraer componentes
    parts = filename.split('_')
    
    if len(parts) >= 5:  # Necesitamos al menos 5 partes
        try:
            # Extraer cada componente según el patrón
            semana = int(parts[0])  # Primer número antes del primer _
            # parts[1] y parts[2] son cadenas que no nos interesan para metadatos
            fecha_str = parts[3]    # Cuarta parte es la fecha (YYYYMMDD)
            numero_clase_raw = parts[4]  # Quinta parte contiene el número de clase
            
            # Extraer solo el número de clase (puede tener sufijos como -220676337)
            numero_clase_match = re.match(r'(\d+)', numero_clase_raw)
            numero_clase = int(numero_clase_match.group(1)) if numero_clase_match else None
            
            # Convertir fecha string a formato legible
            try:
                fecha_obj = datetime.strptime(fecha_str, '%Y%m%d')
                fecha_formateada = fecha_obj.strftime('%Y-%m-%d')
                dia_semana = fecha_obj.strftime('%A')  # Nombre del día en inglés
            except ValueError:
                fecha_formateada = fecha_str
                dia_semana = "Desconocido"
            
            return {
                "semana": semana,
                "fecha": fecha_formateada,
                "fecha_raw": fecha_str,
                "numero_clase": numero_clase,
                "dia_semana": dia_semana
            }
        except (ValueError, IndexError):
            pass
    
    # Si no se puede parsear, devolver valores None
    return {
        "semana": None,
        "fecha": None,
        "fecha_raw": None,
        "numero_clase": None,
        "dia_semana": None
    }

In [21]:
# Prueba de la función extract_file_metadata con patrón basado en guiones bajos
test_files = [
    "6_Semana_AI_20250909_2-220676337.pdf",
    "10_SEMANA_AI_20251007_1-222887296.pdf"
]

print("Prueba de extracción de metadatos con patrón de guiones bajos:")
print("Formato: <semana>_<cadena>_<cadena>_<fecha>_<numero_clase>+<sufijo>")
print("=" * 70)
for filename in test_files:
    metadata = extract_file_metadata(filename)
    print(f"Archivo: {filename}")
    parts = filename.split('_')
    print(f"  Partes: {parts}")
    print(f"  Semana: {metadata['semana']}")
    print(f"  Fecha: {metadata['fecha']}")
    print(f"  Número clase: {metadata['numero_clase']}")
    print(f"  Día semana: {metadata['dia_semana']}")
    print()

Prueba de extracción de metadatos con patrón de guiones bajos:
Formato: <semana>_<cadena>_<cadena>_<fecha>_<numero_clase>+<sufijo>
Archivo: 6_Semana_AI_20250909_2-220676337.pdf
  Partes: ['6', 'Semana', 'AI', '20250909', '2-220676337.pdf']
  Semana: 6
  Fecha: 2025-09-09
  Número clase: 2
  Día semana: Tuesday

Archivo: 10_SEMANA_AI_20251007_1-222887296.pdf
  Partes: ['10', 'SEMANA', 'AI', '20251007', '1-222887296.pdf']
  Semana: 10
  Fecha: 2025-10-07
  Número clase: 1
  Día semana: Tuesday



## Actualización de Metadatos

In [22]:
def update_metadata_in_existing_files(json_folder: Path):
    """
    Extrae autores desde el texto limpio de la primera página de cada JSON.
    Esta función debe ejecutarse DESPUÉS de process_all_pdfs().
    """
    print("=== EXTRACCIÓN DE AUTORES DESDE JSON ===")
    print("Analizando texto limpio de primera página para extraer autores...")
    
    json_files = list(json_folder.glob('*.json'))
    json_files = [f for f in json_files if f.name not in ['docs_metadata.csv']]
    
    if not json_files:
        print(f"No se encontraron archivos JSON en {json_folder}")
        return
    
    print(f"Procesando {len(json_files)} archivos JSON...\n")
    
    updates_made = 0
    authors_found = 0
    
    for json_file in json_files:
        # Cargar el archivo JSON
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        old_author = data['metadata'].get('author', '')
        
        # Solo extraer autor si está como "Desconocido" o vacío
        if old_author == 'Desconocido' or not old_author.strip():
            if data['pages'] and data['pages'][0].get('clean_text'):
                first_page_clean_text = data['pages'][0]['clean_text']
                new_author = extract_author_from_text(first_page_clean_text)
                
                if new_author != "Desconocido":
                    authors_found += 1
                    print(f"[OK] {json_file.name}: '{new_author}'")
                    
                    # Actualizar metadatos
                    data['metadata']['author'] = new_author
                    updates_made += 1
                    
                    # Guardar archivo actualizado
                    with open(json_file, 'w', encoding='utf-8') as f:
                        json.dump(data, f, ensure_ascii=False, indent=2)
                else:
                    print(f"[FAIL] {json_file.name}: No se pudo extraer autor")
            else:
                print(f"[ERROR] {json_file.name}: Sin texto limpio")
        else:
            print(f"→ {json_file.name}: Ya tiene autor: '{old_author}'")
    
    print(f"EXTRACCIÓN COMPLETADA!")
    print(f"   Archivos analizados: {len(json_files)}")
    print(f"   Archivos actualizados: {updates_made}")
    print(f"   Autores encontrados: {authors_found}")
    print(f"   Tasa de éxito: {authors_found/len(json_files)*100:.1f}%")

def preview_metadata_updates(json_folder: Path, max_files: int = 5):
    """
    Muestra una vista previa de qué metadatos se actualizarían sin hacer cambios.
    """
    json_files = list(json_folder.glob('*.json'))[:max_files]
    json_files = [f for f in json_files if f.name not in ['docs_metadata.csv']]
    
    print(f"VISTA PREVIA DE ACTUALIZACIONES DE METADATOS\n")
    
    for json_file in json_files:
        print(f"{json_file.name}")
        
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        current_title = data['metadata'].get('title', '')
        current_author = data['metadata'].get('author', '')
        
        print(f"   Título actual: '{current_title}'")
        print(f"   Autor actual: '{current_author}'")
        
        # Simular actualizaciones
        new_title = current_title if current_title and current_title != json_file.stem else json_file.stem
        
        new_author = current_author
        if not current_author or current_author == 'Desconocido':
            if data['pages']:
                # Usar texto ya limpio si está disponible, sino limpiar el texto crudo
                if 'clean_text' in data['pages'][0]:
                    first_page_clean_text = data['pages'][0]['clean_text']
                else:
                    first_page_clean_text = clean_text(data['pages'][0]['text'])
                new_author = extract_author_from_text(first_page_clean_text)
        
        if new_title != current_title:
            print(f"   Nuevo título: '{new_title}'")
        if new_author != current_author:
            print(f"   Nuevo autor: '{new_author}'")
        
        if new_title == current_title and new_author == current_author:
            print(f"   No requiere cambios")
        
        print()

print("Funciones de actualización de metadatos listas!")

Funciones de actualización de metadatos listas!


## Función Principal

In [23]:
def process_all_pdfs(input_folder: Path, output_folder: Path):
    """
    Procesa todos los PDFs y guarda texto limpio en JSON.
    Los autores se establecen como "Desconocido" para extracción posterior.
    """
    print("=== PROCESAMIENTO PDF A JSON ===")
    print("Extrayendo texto y limpiando contenido...")
    
    docs_metadata = []
    pdf_files = list(input_folder.glob('*.pdf'))
    if not pdf_files:
        print(f"No se encontraron archivos PDF en {input_folder}")
        return

    for pdf_file in pdf_files:
        print(f"Procesando {pdf_file.name}...")
        pdf_data = extract_text_from_pdf(pdf_file)

        # Limpieza del texto por página
        for page in pdf_data['pages']:
            page['clean_text'] = clean_text(page['text'])

        # Guardar JSON
        out_json = output_folder / f"{pdf_file.stem}.json"
        with open(out_json, 'w', encoding='utf-8') as f:
            json.dump(pdf_data, f, ensure_ascii=False, indent=2)

        # Registrar resumen con información extendida
        total_chars = sum(len(p['clean_text']) for p in pdf_data['pages'])
        total_words = sum(len(p['clean_text'].split()) for p in pdf_data['pages'])
        
        # Extraer metadatos del nombre del archivo
        file_metadata = extract_file_metadata(pdf_file.name)
        
        docs_metadata.append({
            "file": pdf_file.name,
            "title": pdf_data['metadata']['title'],
            "author": pdf_data['metadata']['author'],  # Será "Desconocido"
            "semana": file_metadata['semana'],
            "fecha": file_metadata['fecha'],
            "numero_clase": file_metadata['numero_clase'],
            "dia_semana": file_metadata['dia_semana'],
            "pages": pdf_data['num_pages'],
            "chars": total_chars,
            "words": total_words,
            "avg_chars_per_page": round(total_chars / pdf_data['num_pages'], 1) if pdf_data['num_pages'] > 0 else 0
        })

    # Guardar metadatos CSV inicial en la carpeta data/metadata
    metadata_df = pd.DataFrame(docs_metadata)
    metadata_folder = Path("./data/metadata")
    metadata_folder.mkdir(parents=True, exist_ok=True)
    docs_metadata_csv = metadata_folder / "docs_metadata.csv"
    metadata_df.to_csv(docs_metadata_csv, index=False)
    print(f"Metadatos de documentos guardados en {docs_metadata_csv}")
    print(f"PROCESAMIENTO COMPLETADO: {len(pdf_files)} archivos procesados")
    print("Nota: Los autores están como 'Desconocido' - se extraerán en el siguiente paso")

In [24]:
process_all_pdfs(docs_path, output_path)

=== PROCESAMIENTO PDF A JSON ===
Extrayendo texto y limpiando contenido...
Procesando 10_SEMANA_AI_20251007_1-222887296.pdf...
Procesando 10_SEMANA_AI_20251007_1.pdf...
Procesando 10_SEMANA_AI_20251007_1.pdf...
Procesando 10_SEMANA_AI_20251009_1.pdf...
Procesando 10_SEMANA_AI_20251009_1.pdf...
Procesando 11_Semana_AI_20251014_1.pdf...
Procesando 11_Semana_AI_20251014_1.pdf...
Procesando 11_Semana_AI_20251014_2.pdf...
Procesando 11_Semana_AI_20251014_2.pdf...
Procesando 11_Semana_AI_20251014_3.pdf...
Procesando 11_Semana_AI_20251014_3.pdf...
Procesando 11_SEMANA_AI_20251016_2.pdf...
Procesando 11_SEMANA_AI_20251016_2.pdf...
Procesando 11_Semana_AI_20251016_4.pdf...
Procesando 11_Semana_AI_20251016_4.pdf...
Procesando 12_SEMANA_AI_20251021_1.pdf...
Procesando 12_SEMANA_AI_20251021_1.pdf...
Procesando 12_Semana_AI_20251021_2.pdf...
Procesando 12_Semana_AI_20251021_2.pdf...
Procesando 12_SEMANA_AI_20251021_3.pdf...
Procesando 12_SEMANA_AI_20251021_3.pdf...
Procesando 12_SEMANA_AI_20251021_

In [25]:
## Ejecutar Actualización
# Configurar las rutas
output_path = Path("./data/clean_texts")

print("ACTUALIZACIÓN DE METADATOS EN ARCHIVOS EXISTENTES\n")

update_metadata_in_existing_files(output_path)

ACTUALIZACIÓN DE METADATOS EN ARCHIVOS EXISTENTES

=== EXTRACCIÓN DE AUTORES DESDE JSON ===
Analizando texto limpio de primera página para extraer autores...
Procesando 46 archivos JSON...

[OK] 10_SEMANA_AI_20251007_1-222887296.json: 'Gianmarco Oporta Perez'
[OK] 10_SEMANA_AI_20251007_1.json: 'Rodolfo David Acuna Lopez'
[OK] 10_SEMANA_AI_20251009_1.json: 'Luis Felipe Calderon Perez'
[OK] 11_Semana_AI_20251014_1.json: 'Juan Jimenez Valverde'
[OK] 11_Semana_AI_20251014_2.json: 'Luis Fernando Benavides Villegas'
[OK] 11_Semana_AI_20251014_3.json: 'Alex Steven Naranjo Masis'
[OK] 11_SEMANA_AI_20251016_2.json: 'Andres Sanchez Rojas'
[OK] 11_Semana_AI_20251014_1.json: 'Juan Jimenez Valverde'
[OK] 11_Semana_AI_20251014_2.json: 'Luis Fernando Benavides Villegas'
[OK] 11_Semana_AI_20251014_3.json: 'Alex Steven Naranjo Masis'
[OK] 11_SEMANA_AI_20251016_2.json: 'Andres Sanchez Rojas'
[OK] 11_Semana_AI_20251016_4.json: 'Eder Vega Suazo'
[OK] 12_SEMANA_AI_20251021_1.json: 'Andrey Urena Bermudez'
[

In [26]:
def regenerate_docs_metadata_csv(json_folder: Path):
    """Regenera el archivo docs_metadata.csv con información completa de metadatos."""
    json_files = list(json_folder.glob('*.json'))
    json_files = [f for f in json_files if f.name != 'docs_metadata.csv']
    
    if not json_files:
        print(f"No se encontraron archivos JSON en {json_folder}")
        return
    
    print(f"Regenerando docs_metadata.csv con {len(json_files)} archivos...")
    
    docs_metadata = []
    
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Calcular estadísticas
            total_chars = sum(len(p.get('clean_text', '')) for p in data['pages'])
            total_words = sum(len(p.get('clean_text', '').split()) for p in data['pages'])
            num_pages = data['num_pages']
            
            # Extraer metadatos del nombre del archivo
            file_metadata = extract_file_metadata(data['file_name'])
            
            # Agregar información al resumen con metadatos extendidos
            docs_metadata.append({
                "file": data['file_name'],
                "title": data['metadata'].get('title', ''),
                "author": data['metadata'].get('author', 'Desconocido'),
                "semana": file_metadata['semana'],
                "fecha": file_metadata['fecha'],
                "numero_clase": file_metadata['numero_clase'],
                "dia_semana": file_metadata['dia_semana'],
                "pages": num_pages,
                "chars": total_chars,
                "words": total_words,
                "avg_chars_per_page": round(total_chars / num_pages, 1) if num_pages > 0 else 0
            })
            
        except Exception as e:
            print(f"Error procesando {json_file.name}: {e}")
    
    # Crear DataFrame y guardar CSV en data/metadata
    metadata_df = pd.DataFrame(docs_metadata)
    metadata_df = metadata_df.sort_values('file')  # Ordenar por nombre de archivo
    
    metadata_folder = Path("./data/metadata")
    metadata_folder.mkdir(parents=True, exist_ok=True)
    docs_metadata_csv = metadata_folder / "docs_metadata.csv"
    metadata_df.to_csv(docs_metadata_csv, index=False)
    
    print(f"docs_metadata.csv actualizado con {len(docs_metadata)} archivos")
    print(f"Columnas incluidas: {list(metadata_df.columns)}")
    
    # Mostrar estadísticas generales
    print(f"\nEstadísticas generales:")
    print(f"  Total de páginas: {metadata_df['pages'].sum()}")
    print(f"  Total de caracteres: {metadata_df['chars'].sum():,}")
    print(f"  Total de palabras: {metadata_df['words'].sum():,}")
    print(f"  Promedio de páginas por archivo: {metadata_df['pages'].mean():.1f}")
    
    return metadata_df

In [27]:
# Regenerar docs_metadata.csv con metadatos completos
# Los archivos JSON están en output_path, no en output_metadata_path
metadata_df = regenerate_docs_metadata_csv(output_path)

# Mostrar una muestra del resultado
print("\nPrimeras 5 filas de los metadatos de documentos actualizados:")
if metadata_df is not None:
    print(metadata_df.head())
else:
    print("No se pudo generar el DataFrame de metadatos.")

Regenerando docs_metadata.csv con 46 archivos...
docs_metadata.csv actualizado con 46 archivos
Columnas incluidas: ['file', 'title', 'author', 'semana', 'fecha', 'numero_clase', 'dia_semana', 'pages', 'chars', 'words', 'avg_chars_per_page']

Estadísticas generales:
  Total de páginas: 165
  Total de caracteres: 481,704
  Total de palabras: 74,271
  Promedio de páginas por archivo: 3.6

Primeras 5 filas de los metadatos de documentos actualizados:
                                    file                              title  \
0  10_SEMANA_AI_20251007_1-222887296.pdf  10_SEMANA_AI_20251007_1-222887296   
1            10_SEMANA_AI_20251007_1.pdf            10_SEMANA_AI_20251007_1   
2            10_SEMANA_AI_20251009_1.pdf            10_SEMANA_AI_20251009_1   
6            11_SEMANA_AI_20251016_2.pdf            11_SEMANA_AI_20251016_2   
3            11_Semana_AI_20251014_1.pdf            11_Semana_AI_20251014_1   

                       author  semana       fecha  numero_clase dia_semana